# Homework04 - Task 5B Pretrained Inference Only

This notebook runs inference using a pretrained checkpoint and exports node predictions for presentation analysis.

Reference notebooks:
- S0 Classes/S06-15 GML Node Classification.ipynb
- Homework04/Notebooks/05 Homework04_etm_Node_Classification.ipynb

## Cell 2 - Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from topologicpy.PyG import PyG
from topologicpy.Helper import Helper

In [ ]:
print("TopologicPy version:", Helper.Version())

## Cell 5 - Paths

Set your custom dataset path and checkpoint path.

In [ ]:
BASE = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML")
DATASET_PATH = BASE / "Homework04" / "Notebooks" / "dataset_node_classification_custom"
FALLBACK_DATASET = BASE / "example_dataset" / "dataset_node_classification"
CHECKPOINT_PATH = BASE / "S0 Classes" / "msd-main" / "msd_node_classifier.pt"
OUT_DIR = BASE / "Homework04" / "Notebooks" / "classification_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATASET_PATH)
print("Fallback dataset:", FALLBACK_DATASET)
print("Checkpoint path:", CHECKPOINT_PATH)

In [ ]:
def ensure_dataset(path):
    required = [path / "graphs.csv", path / "nodes.csv", path / "edges.csv"]
    ok = all(p.exists() for p in required)
    for p in required:
        print(f"{p.name}: {'FOUND' if p.exists() else 'MISSING'} -> {p}")
    return ok

if not ensure_dataset(DATASET_PATH):
    print("Custom dataset not found. Using fallback reference dataset.")
    DATASET_PATH = FALLBACK_DATASET
    _ = ensure_dataset(DATASET_PATH)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

## Cell 8 - Build PyG object with same task definition

In [ ]:
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level="node",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical"
)

# Keep hyperparameters compatible with S06-15 baseline architecture assumptions
pyg.SetHyperparameters(
    conv="sage",
    hidden_dims=(64, 64, 64),
    activation="relu",
    dropout=0.0,
    batch_norm=True,
    residual=False,
    pooling="mean"
)

print("Dataset loaded for inference.")

## Cell 10 - Load pretrained checkpoint and run inference

Note: this uses the TopologicPy/PyG load and predict methods.

In [ ]:
# Load pretrained model checkpoint
_ = pyg.Load(path=str(CHECKPOINT_PATH))
print("Checkpoint loaded successfully.")

pred_report = pyg.Predict(split="all", return_probs=True, attach_to_data=True)
print("Inference complete.")

## Cell 12 - Export predictions

In [ ]:
rows = []
for gi, data in enumerate(pyg.data_list):
    graph_id = int(data.graph_id.item()) if hasattr(data, "graph_id") else gi
    y_true = np.asarray(pred_report["y_true"][gi]).squeeze()
    y_pred = np.asarray(pred_report["pred"][gi]).squeeze()

    if y_true.ndim > 1:
        y_true = np.argmax(y_true, axis=1)
    if y_pred.ndim > 1:
        y_pred = np.argmax(y_pred, axis=1)

    train_mask = data.train_mask.detach().cpu().numpy() if hasattr(data, "train_mask") else [False] * len(y_true)
    val_mask = data.val_mask.detach().cpu().numpy() if hasattr(data, "val_mask") else [False] * len(y_true)
    test_mask = data.test_mask.detach().cpu().numpy() if hasattr(data, "test_mask") else [False] * len(y_true)

    for ni in range(len(y_true)):
        rows.append({
            "graph_id": graph_id,
            "node_id": int(ni),
            "y_true": int(y_true[ni]),
            "y_pred": int(y_pred[ni]),
            "correct": bool(int(y_true[ni]) == int(y_pred[ni])),
            "train_mask": bool(train_mask[ni]),
            "val_mask": bool(val_mask[ni]),
            "test_mask": bool(test_mask[ni])
        })

pred_df = pd.DataFrame(rows)
pred_csv = OUT_DIR / "node_predictions_pretrained.csv"
pred_df.to_csv(pred_csv, index=False)

print("Saved:", pred_csv)
print("Overall accuracy:", round(pred_df["correct"].mean(), 4))
display(pred_df.head(20))

## Cell 14 - Slide-ready summary template

Use these bullets directly in your presentation:
1. Model source: pretrained checkpoint and dataset used for inference.
2. Overall prediction quality: global and test-split accuracy.
3. Typical correct cases: spatially coherent node classes.
4. Typical incorrect cases: confusion pairs and likely causes.
5. Improvement plan: feature engineering, class balancing, and geometry-aware context.